# Radionica 6: RAG i izgradnja baze znanja

**GirlTHing AI Lab — Blok 2**

> Ćelije označene sa ✏️ **ZADATAK** popunjavate sami. Ostale ćelije samo pokrenite i pročitajte rezultat.

## Ciljevi radionice

1. Razumjeti zašto LLM sam nije dovoljan i šta rješava **RAG** (Retrieval-Augmented Generation).
2. Pretvoriti tekst u **embeddings** (vektore) i mjeriti sličnost.
3. Podijeliti dokumente na **chunkove**.
4. Napraviti **semantičku pretragu** nad bazom znanja.
5. Spojiti pretragu i model u RAG chatbot koji navodi izvore i zna reći "ne znam".

```
Pitanje → embedding → pronađi najsličnije chunkove → [chunkovi + pitanje] → LLM → odgovor s izvorima
```

In [ ]:
# Instalacija paketa (u Colabu je potrebno samo jednom po sesiji)
%pip install -q openai numpy pydantic python-dotenv pypdf

import os
from openai import OpenAI

# API ključ: u Colabu iz "Secrets" (ikona ključa lijevo), lokalno iz .env datoteke
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

client = OpenAI()
MODEL = "gpt-6-luna"     # jeftin i brz model, dovoljan za sve vježbe
print("Spremno ✅")

In [ ]:
import sys
from pathlib import Path

for candidate in [Path('.'), Path('..'), Path('alLab-ml6')]:
    if (candidate / 'data').is_dir() and (candidate / 'rag.py').exists():
        ROOT = candidate.resolve()
        break
else:
    raise FileNotFoundError('Pokrenite notebook iz korijena projekta ili foldera alLab-ml6.')
DATA_DIR = ROOT / 'data'
sys.path.insert(0, str(ROOT))
print('Dokumenti:', sorted(p.name for p in DATA_DIR.iterdir()))

## 1. Problem: model ne zna naše podatke

In [ ]:
r = client.responses.create(model=MODEL, input="Kako se zove završni događaj programa GirlTHing AI Lab i kada se dijele ruksaci?")
print(r.output_text)

Model nije vidio naše dokumente, pa ili ne zna, ili izmišlja. Rješenje: prije pitanja **pronađemo relevantan tekst** i stavimo ga u prompt.

## 2. Embeddings: tekst kao vektor

Embedding model pretvara tekst u vektor od 1.536 brojeva. Tekstovi **sličnog značenja** imaju vektore koji pokazuju u sličnom smjeru, čak i kad ne dijele iste riječi.

Sličnost mjerimo **kosinusnom sličnošću** (sjećate se vektora iz Bloka 1):

$$\text{cos}(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}$$

Vrijednost blizu 1 znači vrlo slično, a blizu 0 nepovezano.

In [ ]:
import numpy as np

EMBED_MODEL = "text-embedding-3-small"

def embed(texts):
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data])

recenice = [
    "Kako da se pripremim za intervju?",
    "Savjeti za razgovor za posao",
    "Koliko dug treba biti CV?",
    "Recept za čokoladnu tortu",
]
V = embed(recenice)
print("Oblik matrice:", V.shape)

### ✏️ ZADATAK 1
Implementiraj funkciju `cosine_similarity(a, b)` pomoću numpy-ja i ispiši sličnost prve rečenice sa svim ostalima.

In [ ]:
def cosine_similarity(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if a.ndim != 1 or b.ndim != 1 or a.shape != b.shape:
        raise ValueError('Očekuju se dva vektora iste dužine.')
    nazivnik = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / nazivnik) if nazivnik else 0.0

for i in range(1, len(recenice)):
    print(f'{cosine_similarity(V[0], V[i]):.3f}  {recenice[0]}  ↔  {recenice[i]}')

💡 "intervju" i "razgovor za posao" nemaju zajedničkih riječi, a ipak su najsličniji. Ovo je **semantička** pretraga, a ne pretraga po ključnim riječima.

## 3. Učitavanje dokumenata

In [ ]:
from pypdf import PdfReader  # za PDF dokumente

def load_documents(folder):
    docs = []
    for path in sorted(Path(folder).glob("*")):
        if path.suffix in (".md", ".txt"):
            docs.append((path.name, path.read_text(encoding="utf-8")))
        elif path.suffix == ".pdf":
            text = "\n".join(p.extract_text() or "" for p in PdfReader(path).pages)
            docs.append((path.name, text))
    return docs

docs = load_documents(DATA_DIR)
for name, text in docs:
    print(f"{name:32} {len(text):5} znakova")

## 4. Chunking: dijeljenje na manje dijelove

Zašto ne šaljemo cijele dokumente?
- Pretraga je preciznija kad je svaki dio o **jednoj temi**.
- Manje teksta u promptu znači **manju cijenu** i manje "šuma" za model.

Najjednostavniji pristup: komadi fiksne dužine s **preklapanjem** (overlap), da rečenica na granici ne izgubi kontekst.

```
|-------- chunk 1 --------|
                  |-------- chunk 2 --------|
                                    |-------- chunk 3 --------|
                  <overlap>
```

### ✏️ ZADATAK 2
Napiši `chunk_text(text, size, overlap)` koja vraća listu komada dužine najviše `size` znakova, gdje svaki sljedeći počinje `size - overlap` znakova nakon prethodnog.

In [ ]:
def chunk_text(text, size=600, overlap=100):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError('Potrebno je size > 0 i 0 <= overlap < size.')
    chunks = []
    for start in range(0, len(text), size - overlap):
        chunks.append(text[start:start + size])
        if start + size >= len(text):
            break
    return chunks

test = chunk_text('abcdefghij' * 20, size=50, overlap=10)
print(len(test), [len(c) for c in test])

In [ ]:
chunks, sources = [], []
for name, text in docs:
    for c in chunk_text(text):
        chunks.append(c)
        sources.append(name)

print("Ukupno chunkova:", len(chunks))
print("\nPrimjer chunka:\n", chunks[3])

🤔 **Diskusija:** chunk iznad možda počinje usred riječi. U `rag.py` koristimo pametniju verziju koja reže po **pasusima**. Kakvi su nedostaci premalih, a kakvi prevelikih chunkova?

## 5. Indeks i pretraga

Izračunamo embedding za svaki chunk jednom i spremimo ih u matricu. To je naša mala **vektorska baza**. Ako vektore normalizujemo na dužinu 1, kosinusna sličnost postaje običan skalarni proizvod, pa sve chunkove poredimo jednim množenjem matrice.

In [ ]:
chunk_vectors = embed(chunks)
chunk_vectors = chunk_vectors / np.linalg.norm(chunk_vectors, axis=1, keepdims=True)
print("Indeks:", chunk_vectors.shape)

### ✏️ ZADATAK 3
Napiši `search(query, k)` koja vraća `k` najsličnijih chunkova kao listu `(sličnost, izvor, tekst)`, sortiranu od najsličnijeg.

*Pomoć:* `np.argsort(-scores)[:k]` daje indekse `k` najvećih vrijednosti.

In [ ]:
def search(query, k=3):
    if k <= 0 or not chunks:
        return []
    q = embed([query])[0]
    norma = np.linalg.norm(q)
    if norma == 0:
        return []
    scores = chunk_vectors @ (q / norma)
    indeksi = np.argsort(-scores, kind='stable')[:k]
    return [(float(scores[i]), sources[i], chunks[i]) for i in indeksi]

for score, src, text in search('Kada da pošaljem zahvalnicu poslije intervjua?'):
    print(f'{score:.3f}  [{src}]  {text[:80]}...')

In [ ]:
for q in ["Kako da učim za ispit iz programiranja?",
          "Šta treba imati GitHub profil?",
          "Ko vodi blok o LLM modelima?",
          "Koja je najbolja pizzerija u Tuzli?"]:
    print("❓", q)
    for score, src, _ in search(q):
        print(f"   {score:.3f}  {src}")

💡 Pogledaj sličnosti za posljednje pitanje (pizzerija). Sve su niske, jer u bazi nema ničeg relevantnog. To koristimo da chatbot zna reći **"ne znam"**.

## 6. RAG: pretraga + generisanje

Pronađene chunkove ubacimo u prompt kao **KONTEKST** i tražimo od modela da odgovara samo na osnovu njega.

In [ ]:
RAG_INSTRUKCIJE = """Ti si Studentski asistent programa GirlTHing AI Lab.
Odgovaraj ISKLJUČIVO na osnovu priloženog KONTEKSTA.
Ako odgovor nije u kontekstu, reci: "Nemam tu informaciju u svojoj bazi znanja."
Na kraju navedi izvore u obliku [naziv_datoteke].
Odgovaraj na bosanskom, kratko i jasno."""

### ✏️ ZADATAK 4
Napiši `rag_answer(question)`:
1. pronađi 4 najsličnija chunka,
2. odbaci one sa sličnošću ispod `MIN_SCORE`,
3. složi kontekst u obliku `[izvor]\ntekst`, odvojen linijama `---`,
4. pozovi model s `RAG_INSTRUKCIJE` i vrati odgovor.

In [ ]:
MIN_SCORE = 0.25

def rag_answer(question):
    hits = [hit for hit in search(question, k=4) if hit[0] >= MIN_SCORE]
    if not hits:
        return 'Nemam tu informaciju u svojoj bazi znanja.'
    kontekst = '\n---\n'.join(f'[{src}]\n{text}' for _, src, text in hits)
    r = client.responses.create(model=MODEL, instructions=RAG_INSTRUKCIJE,
                                input=f'KONTEKST:\n{kontekst}\n\nPITANJE: {question}')
    return r.output_text

print(rag_answer('Kako se zove završni događaj programa GirlTHing AI Lab i šta učesnice dobiju?'))

In [ ]:
for q in ["Šta je STAR metoda?",
          "Koliko projekata da stavim na GitHub?",
          "Koja je najbolja pizzerija u Tuzli?"]:
    print("❓", q)
    print("💬", rag_answer(q), "\n")

### ✏️ ZADATAK 5: Podešavanje praga
Isprobaj `MIN_SCORE` vrijednosti 0.1, 0.25 i 0.45 na pitanjima iznad. Šta se dešava kad je prag prenizak, a šta kad je previsok? Zapiši zaključak u ćeliju ispod.

In [ ]:
pitanja_prag = ['Šta je STAR metoda?', 'Koliko projekata da stavim na GitHub?', 'Koja je najbolja pizzerija u Tuzli?']
prethodni_prag = MIN_SCORE
try:
    for prag in [0.1, 0.25, 0.45]:
        MIN_SCORE = prag
        print(f'\nPRAG {prag}')
        for pitanje in pitanja_prag:
            hits = search(pitanje, k=4)
            print(pitanje, '| prihvaćeno:', sum(s >= prag for s, _, _ in hits))
            print(rag_answer(pitanje))
finally:
    MIN_SCORE = prethodni_prag


**Zaključak:** prag 0.1 propušta više slabije povezanih chunkova, pa u kontekstu može biti više šuma; 0.45 može odbaciti i koristan tekst i povećati broj odgovora „nemam informaciju”. Prag 0.25 je početni kompromis, ali najbolju vrijednost biramo na testnim pitanjima i stvarnim embedding rezultatima. Kosinusna sličnost nije vjerovatnoća tačnog odgovora, a prolazak praga ne garantuje da chunk sadrži odgovor.

## 7. Bonus: vektorska baza ChromaDB

Za veće baze (hiljade dokumenata) koristi se prava vektorska baza. ChromaDB je besplatna i radi lokalno. Logika je ista kao naša: spremi vektore, a zatim traži najsličnije.

In [ ]:
%pip install -q chromadb
import chromadb

chroma = chromadb.Client()                       # baza u memoriji
col = chroma.get_or_create_collection("studentski_asistent", metadata={"hnsw:space": "cosine"})
col.add(ids=[str(i) for i in range(len(chunks))],
        documents=chunks,
        embeddings=chunk_vectors.tolist(),
        metadatas=[{"source": s} for s in sources])

q = embed(["Kada se otvaraju prijave za stipendije?"])[0].tolist()
res = col.query(query_embeddings=[q], n_results=3)
for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(f"{1 - dist:.3f}  [{meta['source']}]  {doc[:70]}...")

## 8. Sve zajedno: `rag.py`

Sve što smo danas napravili nalazi se u datoteci `rag.py`, uz nekoliko poboljšanja: chunking po pasusima, čuvanje indeksa u datoteci (embeddings se računaju samo jednom) i historiju razgovora. Tu datoteku koristimo na sljedećoj radionici za web aplikaciju.

In [ ]:
import rag

index = rag.load_or_build_index(client, DATA_DIR, "index.npz")
odgovor, izvori = rag.answer(client, index, "Šta treba sadržavati LinkedIn profil?")
print(odgovor)
print("Izvori:", izvori)

## 📝 Domaći zadatak

Napravi **test set od 10 pitanja** za bazu znanja Studentskog asistenta (ili za temu koja te zanima), u obliku:

| Pitanje | Očekivani odgovor | Izvor |
| --- | --- | --- |
| Koliko dug treba biti CV? | jedna stranica | pisanje_cv.md |

Uključi **bar 2 pitanja na koja odgovor NIJE u bazi**. Chatbot tada treba reći da ne zna. Test set koristimo na sljedećoj radionici za evaluaciju.

## Sažetak

| Pojam | Ukratko |
| --- | --- |
| RAG | Pretraga relevantnog teksta + generisanje odgovora na osnovu njega |
| Embedding | Vektor koji predstavlja značenje teksta |
| Kosinusna sličnost | Mjera koliko dva vektora "gledaju u istom smjeru" |
| Chunk | Manji dio dokumenta koji se zasebno pretražuje |
| Prag (MIN_SCORE) | Ispod njega smatramo da relevantan izvor ne postoji |

### Rješenje domaće zadaće — 10 evaluacijskih pitanja

| Pitanje | Očekivani odgovor | Izvor |
| --- | --- | --- |
| Koliko dug treba biti CV studentice? | Jednu stranicu. | pisanje_cv.md |
| U kojem formatu poslati CV? | PDF. | pisanje_cv.md |
| Šta znači STAR? | Situacija, zadatak, akcija, rezultat. | razgovor_za_posao.md |
| Kada poslati zahvalnicu nakon intervjua? | U roku od 24 sata. | razgovor_za_posao.md |
| Koliko urednih projekata je dobro imati na GitHubu? | Tri do pet. | github_profil.md |
| Šta treba sadržavati README projekta? | Šta projekat radi, kako se pokreće i koje tehnologije koristi; snimak ekrana ili primjer rezultata. | github_profil.md |
| Šta napisati u LinkedIn About? | Tri do pet rečenica o studiju, interesovanjima i prilici koju tražite. | linkedin_profil.md |
| Kako se zove završni događaj i kada se dijele ruksaci? | AI Challenge; nakon proglašenja pobjednica. | girlthing_ai_lab.md |
| Koja je najbolja pizzerija u Tuzli? | Nemam tu informaciju u svojoj bazi znanja. | Nema u bazi |
| Koji je tačan datum AI Challengea? | Nemam tu informaciju u svojoj bazi znanja. | Nema u bazi |

Za pitanja bez odgovora provjeriti da model odbije nagađanje. Za ostala provjeriti tačnost odgovora i navedeni izvor; prisustvo riječi samo po sebi nije dovoljna evaluacija.